In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [12]:
import os

print("Current Directory:", os.getcwd())
print("Files here:", os.listdir())

Current Directory: c:\Users\User\Downloads\sales-analysis-dashboard\2_notebooks
Files here: ['salesanalysis.ipynb']


In [14]:
import os
print(os.getcwd())

c:\Users\User\Downloads\sales-analysis-dashboard\2_notebooks


In [15]:
print(os.listdir())

['salesanalysis.ipynb']


In [16]:
print(os.listdir(".."))

['.git', '.gitignore', '1_data', '2_notebooks', '3_dashboards', 'assets', 'README.md', 'requirements.txt', 'utils', 'venv']


In [18]:
import pandas as pd

# 1. Use sep=None and engine='python' to auto-detect the format
# 2. Use on_bad_lines='skip' to ignore rows that don't match the structure
df = pd.read_csv("../1_data/raw/salesdata.csv", sep=None, engine='python', on_bad_lines='skip')

# Display the results
print(df.head())
print(df.info())
print(df.describe())


          Category         City        Country Customer.ID     Customer.Name  \
0  Office Supplies  Los Angeles  United States   LS-172304  Lycoris Saunders   
1  Office Supplies  Los Angeles  United States   MV-174854     Mark Van Huff   
2  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
3  Office Supplies  Los Angeles  United States   CS-121304      Chad Sievert   
4  Office Supplies  Los Angeles  United States   AP-109154    Arthur Prichep   

   Discount Market Order.Date        Order.ID Order.Priority  ... Quantity  \
0         0     US    00:00.0  CA-2011-130813           High  ...        3   
1         0     US    00:00.0  CA-2011-148614         Medium  ...        2   
2         0     US    00:00.0  CA-2011-118962         Medium  ...        3   
3         0     US    00:00.0  CA-2011-118962         Medium  ...        2   
4         0     US    00:00.0  CA-2011-146969           High  ...        1   

  Region  Sales  Ship.Date       Ship.Mode  Shippi

In [19]:
# Convert date
df["Order.Date"] = pd.to_datetime(df["Order.Date"])

# Check nulls
df.isnull().sum()

# Remove duplicates
df.drop_duplicates(inplace=True)

# Standardize column names
df.columns = df.columns.str.strip().str.replace(" ", ".")

In [20]:
print(os.listdir(".."))

['.git', '.gitignore', '1_data', '2_notebooks', '3_dashboards', 'assets', 'README.md', 'requirements.txt', 'utils', 'venv']


In [21]:
# Total sales & profit
print("Total Sales:", df["Sales"].sum())
print("Total Profit:", df["Profit"].sum())

# Region-wise sales
df.groupby("Region")["Sales"].sum()

# Category performance
df.groupby("Category")[["Sales","Profit"]].sum()

Total Sales: 1283051
Total Profit: 342112.8264


,Sales,Profit
Category,,
Office Supplies,1283051,342112.8264


In [22]:
df["Year"] = df["Order.Date"].dt.year
df["Month"] = df["Order.Date"].dt.month

monthly_sales = df.groupby(["Year","Month"])["Sales"].sum().reset_index()

In [24]:
# Displays a list of all column names
print(df.columns.tolist())


['Category', 'City', 'Country', 'Customer.ID', 'Customer.Name', 'Discount', 'Market', 'Order.Date', 'Order.ID', 'Order.Priority', 'Product.ID', 'Product.Name', 'Profit', 'Quantity', 'Region', 'Sales', 'Ship.Date', 'Ship.Mode', 'Shipping.Cost', 'State', 'Sub.Category', 'Year', 'Revenue', 'Month']


In [27]:
# Top region
df.groupby("Region")["Sales"].sum().sort_values(ascending=True)

# Worst category (profit)
df.groupby("Category")["Profit"].sum().sort_values()

df.groupby("Market")["Sales"].sum().sort_values(ascending=True)

Market
Canada     14356
EMEA      100305
Africa    111120
APAC      165056
LATAM     188856
EU        261151
US        442207
Name: Sales, dtype: int64

In [28]:
# Sub-category profit
subcat_profit = df.groupby("Sub.Category")["Profit"].sum().sort_values()

subcat_profit.head(10)

Sub.Category
Fasteners      6617.0152
Labels         9187.4860
Supplies      16526.3342
Envelopes     18295.6644
Art           34287.2406
Paper         38498.7777
Binders       60426.8807
Storage       71556.4615
Appliances    86716.9661
Name: Profit, dtype: float64

In [29]:
loss_products = df[df["Profit"] < 0]

loss_products[["Product.Name","Sales","Profit"]].head(10)

,Product.Name,Sales,Profit


In [30]:
df[["Sales","Profit"]].corr()

,Sales,Profit
Sales,1.000000,0.857891
Profit,0.857891,1.000000


In [31]:
df.groupby("Market")["Sales"].sum().sort_values(ascending=False)

Market
US        442207
EU        261151
LATAM     188856
APAC      165056
Africa    111120
EMEA      100305
Canada     14356
Name: Sales, dtype: int64

In [32]:
forecast_df = df.groupby(pd.Grouper(key='Order.Date', freq='M'))['Sales'].sum().reset_index()

forecast_df.columns = ["ds", "y"]

In [33]:
df["Profit.Margin"] = df["Profit"] / df["Sales"]
df["Order.Value"] = df["Sales"]

In [34]:
df.to_csv("../1_data/processed/cleaned_sales.csv", index=False)